In [ ]:
!pip install transformers ipywidgets pdfkit pillow folium faker gtts googletrans==4.0.0-rc1 beautifulsoup4
!apt-get install -y wkhtmltopdf

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from transformers import pipeline
import ipywidgets as widgets
from IPython.display import display, FileLink, Image as IPyImage, Audio
import pdfkit
from google.colab import files
from PIL import Image, ImageDraw, ImageFont
import folium
from faker import Faker
import datetime
import random
from gtts import gTTS
from googletrans import Translator
from bs4 import BeautifulSoup
import os
import json

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-large")
fake = Faker()
translator = Translator()

In [ ]:
def classify_scenario(scenario):
    scenario = scenario.lower()
    if any(word in scenario for word in ["sick", "illness", "fever", "doctor", "medical", "stomach", "bathroom", "unwell"]):
        return "medical"
    elif any(word in scenario for word in ["school", "exam", "child", "homework"]):
        return "parental"
    elif any(word in scenario for word in ["work", "office", "meeting", "hr", "manager", "boss"]):
        return "official"
    elif any(word in scenario for word in ["travel", "flight", "train", "delay", "location", "traffic", "rain"]):
        return "travel"
    else:
        return "none"

def generate_document_html(doc_type, doc_body, lang_code="en"):
    today = datetime.date.today().strftime("%Y-%m-%d")
    if doc_type == "medical":
        return f"""
        <html>
        <head>
        <style>
          body {{ font-family: Arial, sans-serif; font-size: 14pt; margin: 40px; }}
          .signature {{ margin-top: 40px; }}
        </style>
        </head>
        <body>
          <h2 style="text-align: center;">Medical Certificate</h2>
          <p><b>Date:</b> {today}</p>
          <p>To whom it may concern,</p>
          <p>{doc_body}</p>
          <div class="signature">
            <p>Doctor: Dr. {fake.last_name()}</p>
            <p>Signature: ___________________</p>
          </div>
        </body>
        </html>
        """
    elif doc_type == "parental":
        return f"""
        <html>
        <head>
        <style>
          body {{ font-family: Arial, sans-serif; font-size: 14pt; margin: 40px; }}
          .signature {{ margin-top: 40px; }}
        </style>
        </head>
        <body>
          <h2 style="text-align: center;">Parental Note</h2>
          <p><b>Date:</b> {today}</p>
          <p>To whom it may concern,</p>
          <p>{doc_body}</p>
          <div class="signature">
            <p>Parent/Guardian Signature: ___________________</p>
          </div>
        </body>
        </html>
        """
    elif doc_type == "official":
        return f"""
        <html>
        <head>
        <style>
          body {{ font-family: Arial, sans-serif; font-size: 14pt; margin: 40px; }}
          .signature {{ margin-top: 40px; }}
        </style>
        </head>
        <body>
          <h2 style="text-align: center;">Official Work Excuse</h2>
          <p><b>Date:</b> {today}</p>
          <p>To whom it may concern,</p>
          <p>{doc_body}</p>
          <div class="signature">
            <p>Manager Signature: ___________________</p>
          </div>
        </body>
        </html>
        """
    elif doc_type == "travel":
        return f"""
        <html>
        <head>
        <style>
          body {{ font-family: Arial, sans-serif; font-size: 14pt; margin: 40px; }}
          .signature {{ margin-top: 40px; }}
        </style>
        </head>
        <body>
          <h2 style="text-align: center;">Travel Delay Certificate</h2>
          <p><b>Date:</b> {today}</p>
          <p>{doc_body}</p>
          <div class="signature">
            <p>Transport Authority Signature: ___________________</p>
          </div>
        </body>
        </html>
        """
    else:
        return None

def generate_fake_chat_image(scenario, excuse_text, filename="chat_screenshot.png", lang_text="en"):
    if "medical" in scenario:
        user = "Dr. Smith"
        messages = [
            ("Dr. Smith", "How are you feeling today?"),
            ("Patient", excuse_text)
        ]
    elif "parental" in scenario or "school" in scenario:
        user = "Teacher"
        messages = [
            ("Teacher", "Your child was absent today. Please send a note."),
            ("Parent", excuse_text)
        ]
    elif "official" in scenario:
        user = "Manager"
        messages = [
            ("Manager", "Please explain your absence from the meeting."),
            ("Employee", excuse_text)
        ]
    else:
        user = "Friend"
        messages = [
            ("Friend", "What happened?"),
            ("You", excuse_text)
        ]
    if lang_text != "en":
        messages = [(sender, translator.translate(msg, dest=lang_text).text) for sender, msg in messages]
    width, height = 600, 200 + 30 * len(messages)
    img = Image.new('RGB', (width, height), color=(255, 255, 255))
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    d.text((10, 10), f"Chat with {user}", fill=(0, 0, 0), font=font)
    y_text = 40
    for sender, message in messages:
        d.text((10, y_text), f"{sender}: {message}", fill=(0, 0, 0), font=font)
        y_text += 30
    img.save(filename)
    return filename

def generate_fake_location_map(filename="fake_location_map.html"):
    latitude = fake.latitude()
    longitude = fake.longitude()
    mymap = folium.Map(location=[latitude, longitude], zoom_start=15)
    folium.Marker([latitude, longitude], popup="Fake Location").add_to(mymap)
    mymap.save(filename)
    return filename, latitude, longitude

In [ ]:
HISTORY_FILE = "/content/excuse_history.json"
FAVORITES_FILE = "/content/excuse_favorites.json"
RANKINGS_FILE = "/content/excuse_rankings.json"

def load_list(filename):
    if os.path.exists(filename):
        with open(filename, "r") as f:
            return json.load(f)
    return []

def save_list(filename, data):
    with open(filename, "w") as f:
        json.dump(data, f)

def add_to_history(excuse, scenario, lang):
    history = load_list(HISTORY_FILE)
    history.append({
        "excuse": excuse,
        "scenario": scenario,
        "lang": lang,
        "timestamp": datetime.datetime.now().isoformat()
    })
    save_list(HISTORY_FILE, history)

def add_to_favorites(excuse, scenario, lang):
    favorites = load_list(FAVORITES_FILE)
    if not any(f["excuse"] == excuse and f["scenario"] == scenario and f["lang"] == lang for f in favorites):
        favorites.append({
            "excuse": excuse,
            "scenario": scenario,
            "lang": lang,
            "timestamp": datetime.datetime.now().isoformat()
        })
        save_list(FAVORITES_FILE, favorites)

def remove_from_favorites(excuse, scenario, lang):
    favorites = load_list(FAVORITES_FILE)
    favorites = [f for f in favorites if not (f["excuse"] == excuse and f["scenario"] == scenario and f["lang"] == lang)]
    save_list(FAVORITES_FILE, favorites)

def add_ranking(excuse, scenario, lang, score):
    rankings = load_list(RANKINGS_FILE)
    rankings.append({
        "excuse": excuse,
        "scenario": scenario,
        "lang": lang,
        "score": score,
        "timestamp": datetime.datetime.now().isoformat()
    })
    save_list(RANKINGS_FILE, rankings)

def get_ranked_excuses(scenario, lang, top_n=3):
    rankings = load_list(RANKINGS_FILE)
    # Filter by scenario and language, then sort by average score
    filtered = [r for r in rankings if r["scenario"].lower() == scenario.lower() and r["lang"] == lang]
    if not filtered:
        return []
    # Aggregate scores by excuse
    score_dict = {}
    count_dict = {}
    for r in filtered:
        key = r["excuse"]
        score_dict[key] = score_dict.get(key, 0) + r["score"]
        count_dict[key] = count_dict.get(key, 0) + 1
    avg_scores = [(excuse, score_dict[excuse]/count_dict[excuse]) for excuse in score_dict]
    avg_scores.sort(key=lambda x: x[1], reverse=True)
    return avg_scores[:top_n]

In [ ]:
history_area = widgets.Output()
favorites_area = widgets.Output()
ranking_area = widgets.Output()

def show_history():
    history_area.clear_output()
    with history_area:
        history = load_list(HISTORY_FILE)
        if not history:
            print("No excuse history yet.")
            return
        for i, entry in enumerate(reversed(history[-10:]), 1):
            print(f"{i}. [{entry['lang']}] {entry['scenario']}\n   {entry['excuse']}\n")

def show_favorites():
    favorites_area.clear_output()
    with favorites_area:
        favorites = load_list(FAVORITES_FILE)
        if not favorites:
            print("No favorites saved yet.")
            return
        for i, entry in enumerate(reversed(favorites), 1):
            print(f"{i}. [{entry['lang']}] {entry['scenario']}\n   {entry['excuse']}\n")

def show_rankings(scenario, lang):
    ranking_area.clear_output()
    with ranking_area:
        ranked = get_ranked_excuses(scenario, lang)
        if not ranked:
            print("No ranked excuses yet for this scenario/language.")
        else:
            print("Top Ranked Excuses:")
            for i, (excuse, avg_score) in enumerate(ranked, 1):
                print(f"{i}. {excuse}\n   (Avg. Effectiveness Score: {avg_score:.2f})\n")

In [ ]:
LANGUAGES = {
    "English": "en",
    "Hindi": "hi",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Chinese": "zh-cn",
    "Arabic": "ar"
}

excuse_context = widgets.Text(
    placeholder='Enter scenario (e.g. "late for work")',
    description='Scenario:'
)
lang_selector = widgets.Dropdown(
    options=LANGUAGES,
    value="en",
    description='Language:'
)
generate_btn = widgets.Button(description="Generate Excuse & Proof")
save_fav_btn = widgets.Button(description="Save to Favorites", button_style='success')
show_history_btn = widgets.Button(description="Show History")
show_fav_btn = widgets.Button(description="Show Favorites")
rate_excuse_btn = widgets.Button(description="Rate Excuse", button_style='info')
show_ranking_btn = widgets.Button(description="Show Ranked Excuses")
rating_selector = widgets.Dropdown(
    options=[("Very Effective", 5), ("Effective", 4), ("Neutral", 3), ("Ineffective", 2), ("Very Ineffective", 1)],
    value=5,
    description='Rate:'
)
output_area = widgets.Output()

last_excuse = {"excuse": "", "scenario": "", "lang": ""}

def generate_excuse(_):
    with output_area:
        output_area.clear_output()
        scenario = excuse_context.value.strip()
        lang_code = lang_selector.value
        if not scenario:
            print("⚠️ Please enter a scenario first")
            return

        doc_type = classify_scenario(scenario)
        prompt = (
            f"Write a highly believable, detailed, scenario-specific excuse for: '{scenario}'. "
            "Make sure the excuse logically fits the scenario and, if needed, add a brief explanation. "
            "Include realistic details and avoid generic or cliché responses. "
            f"Tone: {'Professional' if doc_type in ['official', 'medical'] else 'Casual'}."
        )
        try:
            result = generator(prompt, max_new_tokens=100)[0]['generated_text']
            if lang_code != "en":
                translated_excuse = translator.translate(result, dest=lang_code).text
            else:
                translated_excuse = result
            print(f"🚀 Your AI-generated excuse:\n\n{translated_excuse}\n")

            add_to_history(translated_excuse, scenario, lang_code)
            last_excuse["excuse"] = translated_excuse
            last_excuse["scenario"] = scenario
            last_excuse["lang"] = lang_code

            tts = gTTS(translated_excuse, lang=lang_code)
            audio_path = "/content/excuse_audio.mp3"
            tts.save(audio_path)
            print("🔊 Playing the excuse as audio:")
            display(Audio(audio_path, autoplay=True))
            files.download(audio_path)

            doc_body = None
            if doc_type != "none":
                doc_prompt = (
                    f"Write a formal, scenario-specific note or certificate for this situation: '{scenario}'. "
                    "Make it realistic and relevant, as a professional would write. "
                    "Do not just repeat the excuse; provide supporting details."
                )
                doc_body = generator(doc_prompt, max_new_tokens=120)[0]['generated_text']
                if lang_code != "en":
                    doc_body = translator.translate(doc_body, dest=lang_code).text

            doc_html = generate_document_html(doc_type, doc_body, lang_code) if doc_body else None
            scenario_type = doc_type

            if doc_html:
                if lang_code != "en":
                    soup = BeautifulSoup(doc_html, "html.parser")
                    for tag in soup.find_all("p"):
                        tag.string = translator.translate(tag.text, dest=lang_code).text
                    doc_html = str(soup)
                pdf_path = "/content/supporting_document.pdf"
                options = {'encoding': 'UTF-8'}
                pdfkit.from_string(doc_html, pdf_path, options=options)
                print(f"📄 Supporting PDF generated: {pdf_path}")
                files.download(pdf_path)

            if scenario_type in ["medical", "parental", "official"]:
                chat_path = "/content/chat_screenshot.png"
                generate_fake_chat_image(scenario_type, translated_excuse, chat_path, lang_text=lang_code)
                print("💬 Fake chat screenshot:")
                display(IPyImage(chat_path))
                files.download(chat_path)

            if scenario_type == "travel":
                map_path, lat, lon = generate_fake_location_map()
                print(f"📍 Fake location: {lat}, {lon}")
                display(FileLink(map_path, result_html_prefix="Download fake location map: "))
                files.download(map_path)

            if scenario_type == "none":
                print("No supporting document, chat, or location proof is needed for this scenario.")
        except Exception as e:
            print(f"❌ Error: {str(e)}")

def save_to_favorites(_):
    if last_excuse["excuse"]:
        add_to_favorites(last_excuse["excuse"], last_excuse["scenario"], last_excuse["lang"])
        print("⭐ Excuse added to favorites!")

def rate_excuse(_):
    if last_excuse["excuse"]:
        add_ranking(last_excuse["excuse"], last_excuse["scenario"], last_excuse["lang"], rating_selector.value)
        print("✅ Your feedback has been recorded.")

def show_history_handler(_):
    show_history()

def show_favorites_handler(_):
    show_favorites()

def show_ranking_handler(_):
    scenario = excuse_context.value.strip()
    lang_code = lang_selector.value
    show_rankings(scenario, lang_code)

generate_btn.on_click(generate_excuse)
save_fav_btn.on_click(save_to_favorites)
rate_excuse_btn.on_click(rate_excuse)
show_history_btn.on_click(show_history_handler)
show_fav_btn.on_click(show_favorites_handler)
show_ranking_btn.on_click(show_ranking_handler)

display(
    widgets.VBox([
        widgets.HBox([excuse_context, lang_selector]),
        widgets.HBox([generate_btn, save_fav_btn, show_history_btn, show_fav_btn]),
        widgets.HBox([rating_selector, rate_excuse_btn, show_ranking_btn]),
        output_area,
        widgets.Label("Excuse History (last 10):"), history_area,
        widgets.Label("Favorites:"), favorites_area,
        widgets.Label("Smart Excuse Ranking:"), ranking_area
    ])
)